In [0]:
# Record current swf_version and note timings before we modify them
from pyspark.sql.functions import col

test_song_id = 3251

print("Recording baseline state...")
print("="*70)

if spark.catalog.tableExists("acubed.ffr.bronze__songlist"):
    test_song = spark.table("acubed.ffr.bronze__songlist").filter(col("id") == test_song_id)
    
    if test_song.count() > 0:
        song_data = test_song.select("id", "name", "swf_version").first()
        original_version = song_data.swf_version
        
        print(f"Current state of song {test_song_id}:")
        print(f"  Name: {song_data.name}")
        print(f"  swf_version: {original_version}")
        print(f"\n✓ Recorded original swf_version: {original_version}")
        
        # Store it in a temp view for reference
        spark.sql(f"""CREATE OR REPLACE TEMP VIEW original_song_state AS 
                      SELECT {test_song_id} as song_id, '{original_version}' as original_swf_version""")
        print(f"✓ Stored in temp view: original_song_state")
        
        # Also check note timings
        if spark.catalog.tableExists("acubed.ffr.bronze__charts"):
            note_sample = spark.sql(f"""
                SELECT song_id, info['difficulty'] as difficulty, info['note_count'] as note_count
                FROM acubed.ffr.bronze__charts
                WHERE song_id = {test_song_id}
                LIMIT 3
            """)
            if note_sample.count() > 0:
                print(f"\n✓ Note data exists in bronze__charts")
                print(f"  Charts for song {test_song_id}:")
                note_sample.show(truncate=False)
        
        print(f"\n📋 Ready to proceed to Step 2")
    else:
        print(f"❌ Song {test_song_id} not found in bronze__songlist")
        print(f"\n⚠️ PREREQUISITE MISSING")
        print(f"   Run Scenario 1 first to add song {test_song_id}")
else:
    print("❌ bronze__songlist does not exist")
    print("\n⚠️ PREREQUISITE MISSING")
    print("   Run the bronze ingestion notebook first to create baseline data")

In [0]:
# Modify data in ALL bronze, silver, and gold tables to simulate stale data
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, expr, when

test_song_id = 3251
test_song_name = "Lisrim"
fake_version = 999999  # swf_version is BIGINT
fake_note_count = 99999  # For testing note_count changes

print("="*70)
print("MODIFYING DATA TO SIMULATE STALE STATE")
print("="*70)
print(f"\nTest song: {test_song_id} ({test_song_name})")
print(f"Purpose: Simulate stale data across ALL incremental tables\n")

tables_modified = 0
tables_skipped = 0

# ============================================================================
# BRONZE LAYER MODIFICATIONS
# ============================================================================

print("\n[BRONZE LAYER]")
print("-" * 70)

# Bronze: songlist - modify swf_version (PRIMARY TEST TARGET)
if spark.catalog.tableExists("acubed.ffr.bronze__songlist"):
    original = spark.table("acubed.ffr.bronze__songlist").filter(col("id") == test_song_id).select("swf_version").first()
    
    if original:
        original_version = original.swf_version
        
        spark.sql(f"""
        UPDATE acubed.ffr.bronze__songlist
        SET swf_version = {fake_version}
        WHERE id = {test_song_id}
        """)
        
        print(f"✓ bronze__songlist - swf_version modified")
        print(f"  Before: {original_version}")
        print(f"  After: {fake_version}")
        tables_modified += 1
    else:
        print(f"○ bronze__songlist - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ bronze__songlist - table doesn't exist (skip)")
    tables_skipped += 1

# Bronze: charts - modify note_count in map structure using DataFrame API
if spark.catalog.tableExists("acubed.ffr.bronze__charts"):
    charts = spark.table("acubed.ffr.bronze__charts").filter(col("song_id") == test_song_id)
    
    if charts.count() > 0:
        # Use Delta merge to update the map with modified note_count
        deltaTable = DeltaTable.forName(spark, "acubed.ffr.bronze__charts")
        
        # Create a temp DataFrame with the updated info map
        updated_charts = (
            spark.table("acubed.ffr.bronze__charts")
            .filter(col("song_id") == test_song_id)
            .withColumn(
                "info",
                expr(f"map_from_entries(transform(map_entries(info), x -> if(x.key = 'note_count', struct(x.key as key, '{fake_note_count}' as value), x)))")
            )
        )
        
        # Merge back
        deltaTable.alias("target").merge(
            updated_charts.alias("source"),
            "target.song_id = source.song_id"
        ).whenMatchedUpdate(
            set={"info": "source.info"}
        ).execute()
        
        print(f"✓ bronze__charts - info['note_count'] modified")
        print(f"  Changed to: {fake_note_count}")
        tables_modified += 1
    else:
        print(f"○ bronze__charts - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ bronze__charts - table doesn't exist (skip)")
    tables_skipped += 1

# Bronze: contested-difficulties
if spark.catalog.tableExists("acubed.ffr.`bronze__contested-difficulties`"):
    contested = spark.table("acubed.ffr.`bronze__contested-difficulties`").filter(col("song_name") == test_song_name)
    
    if contested.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`bronze__contested-difficulties`
        SET new_diff = 999, old_diff = 999
        WHERE song_name = '{test_song_name}'
        """)
        
        print(f"✓ bronze__contested-difficulties - new_diff/old_diff modified")
        print(f"  Changed to: 999")
        tables_modified += 1
    else:
        print(f"○ bronze__contested-difficulties - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ bronze__contested-difficulties - table doesn't exist (skip)")
    tables_skipped += 1

# ============================================================================
# SILVER LAYER MODIFICATIONS
# ============================================================================

print("\n[SILVER LAYER]")
print("-" * 70)

# Silver: chart-features - modify swf_version
if spark.catalog.tableExists("acubed.ffr.`silver__chart-features`"):
    features = spark.table("acubed.ffr.`silver__chart-features`").filter(col("song_id") == test_song_id)
    
    if features.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`silver__chart-features`
        SET swf_version = {fake_version}
        WHERE song_id = {test_song_id}
        """)
        
        print(f"✓ silver__chart-features - swf_version modified")
        print(f"  Changed to: {fake_version}")
        tables_modified += 1
    else:
        print(f"○ silver__chart-features - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__chart-features - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: notes - modify note timestamp
if spark.catalog.tableExists("acubed.ffr.silver__notes"):
    notes = spark.table("acubed.ffr.silver__notes").filter(col("song_id") == test_song_id)
    
    if notes.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.silver__notes
        SET timestamp = timestamp + 999999
        WHERE song_id = {test_song_id}
        """)
        
        count = notes.count()
        print(f"✓ silver__notes - timestamp modified for {count} notes")
        print(f"  Added: 999999 to all note timestamps")
        tables_modified += 1
    else:
        print(f"○ silver__notes - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__notes - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: notes-adjusted - modify adjusted note timestamp
if spark.catalog.tableExists("acubed.ffr.`silver__notes-adjusted`"):
    notes_adj = spark.table("acubed.ffr.`silver__notes-adjusted`").filter(col("song_id") == test_song_id)
    
    if notes_adj.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`silver__notes-adjusted`
        SET timestamp = timestamp + 999999
        WHERE song_id = {test_song_id}
        """)
        
        count = notes_adj.count()
        print(f"✓ silver__notes-adjusted - timestamp modified for {count} notes")
        print(f"  Added: 999999 to all adjusted timestamps")
        tables_modified += 1
    else:
        print(f"○ silver__notes-adjusted - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__notes-adjusted - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: vertical-density - modify timestamp values
if spark.catalog.tableExists("acubed.ffr.`silver__vertical-density`"):
    vdensity = spark.table("acubed.ffr.`silver__vertical-density`").filter(col("song_id") == test_song_id)
    
    if vdensity.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`silver__vertical-density`
        SET timestamp = timestamp + 999999
        WHERE song_id = {test_song_id}
        """)
        
        count = vdensity.count()
        print(f"✓ silver__vertical-density - timestamp modified for {count} records")
        print(f"  Added: 999999 to all timestamps")
        tables_modified += 1
    else:
        print(f"○ silver__vertical-density - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__vertical-density - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: horizontal-density - modify timestamp values
if spark.catalog.tableExists("acubed.ffr.`silver__horizontal-density`"):
    hdensity = spark.table("acubed.ffr.`silver__horizontal-density`").filter(col("song_id") == test_song_id)
    
    if hdensity.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`silver__horizontal-density`
        SET timestamp = timestamp + 999999
        WHERE song_id = {test_song_id}
        """)
        
        count = hdensity.count()
        print(f"✓ silver__horizontal-density - timestamp modified for {count} records")
        print(f"  Added: 999999 to all timestamps")
        tables_modified += 1
    else:
        print(f"○ silver__horizontal-density - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__horizontal-density - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: playlist - modify song metadata if present
if spark.catalog.tableExists("acubed.ffr.silver__playlist"):
    playlist = spark.table("acubed.ffr.silver__playlist").filter(col("song_id") == test_song_id)
    
    if playlist.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.silver__playlist
        SET swf_version = {fake_version}
        WHERE song_id = {test_song_id}
        """)
        
        print(f"✓ silver__playlist - swf_version modified")
        print(f"  Changed to: {fake_version}")
        tables_modified += 1
    else:
        print(f"○ silver__playlist - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__playlist - table doesn't exist (skip)")
    tables_skipped += 1

# Silver: contested-difficulties
if spark.catalog.tableExists("acubed.ffr.`silver__contested-difficulties`"):
    silver_contested = spark.table("acubed.ffr.`silver__contested-difficulties`").filter(col("song_name") == test_song_name)
    
    if silver_contested.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`silver__contested-difficulties`
        SET new_difficulty = 999
        WHERE song_name = '{test_song_name}'
        """)
        
        print(f"✓ silver__contested-difficulties - new_difficulty modified")
        print(f"  Changed to: 999")
        tables_modified += 1
    else:
        print(f"○ silver__contested-difficulties - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ silver__contested-difficulties - table doesn't exist (skip)")
    tables_skipped += 1

# ============================================================================
# GOLD LAYER MODIFICATIONS
# ============================================================================

print("\n[GOLD LAYER]")
print("-" * 70)

# Gold: song-features - modify swf_version
if spark.catalog.tableExists("acubed.ffr.`gold__features`"):
    gold_features = spark.table("acubed.ffr.`gold__features`").filter(col("song_id") == test_song_id)
    
    if gold_features.count() > 0:
        spark.sql(f"""
        UPDATE acubed.ffr.`gold__features`
        SET swf_version = {fake_version}
        WHERE song_id = {test_song_id}
        """)
        
        print(f"✓ gold__features - swf_version modified")
        print(f"  Changed to: {fake_version}")
        tables_modified += 1
    else:
        print(f"○ gold__features - song not present (skip)")
        tables_skipped += 1
else:
    print(f"⊘ gold__features - table doesn't exist (skip)")
    tables_skipped += 1

print("\n" + "="*70)
print("MODIFICATION SUMMARY")
print("="*70)
print(f"Tables modified: {tables_modified}")
print(f"Tables skipped: {tables_skipped}")
print(f"\n✅ Preparation complete - {tables_modified} tables modified")
print(f"\nSong {test_song_id} data set to fake values across bronze/silver/gold layers")
print(f"Bronze ingestion will detect changes and UPDATE to correct API values")
print(f"Silver layer will propagate updates when silver notebooks are run")
print(f"Gold layer will propagate updates when gold notebooks are run")
print(f"\nReady to run bronze ingestion notebook for comprehensive end-to-end testing")

In [0]:
# Run this AFTER running the bronze ingestion notebook AND silver/gold notebooks
# Verify that song 3251 updates were detected and applied across all layers
from pyspark.sql.functions import col

test_song_id = 3251
test_song_name = "Lisrim"
fake_version = 999999  # Must match the value from Step 2

print("="*70)
print("UPDATE DETECTION VERIFICATION")
print("="*70)
print(f"\nChecking if modifications to song {test_song_id} were detected and corrected...\n")

# Define all testable tables to verify across all layers
tables_to_verify = [
    # Bronze layer - should be updated after bronze ingestion
    ("acubed.ffr.bronze__songlist", "id", "swf_version", "bronze"),
    ("acubed.ffr.bronze__charts", "song_id", None, "bronze"),
    ("acubed.ffr.`bronze__contested-difficulties`", "song_name", "new_diff", "bronze"),
    
    # Silver layer - should be updated after silver notebooks run
    ("acubed.ffr.`silver__chart-features`", "song_id", "swf_version", "silver"),
    ("acubed.ffr.silver__notes", "song_id", "timestamp", "silver"),
    ("acubed.ffr.`silver__notes-adjusted`", "song_id", "timestamp", "silver"),
    ("acubed.ffr.`silver__vertical-density`", "song_id", "timestamp", "silver"),
    ("acubed.ffr.`silver__horizontal-density`", "song_id", "timestamp", "silver"),
    ("acubed.ffr.silver__playlist", "song_id", "swf_version", "silver"),
    ("acubed.ffr.`silver__contested-difficulties`", "song_name", "new_difficulty", "silver"),
    
    # Gold layer - should be updated after gold notebooks run
    ("acubed.ffr.`gold__features`", "song_id", "swf_version", "gold"),
]

success_count = 0
failure_count = 0
pending_bronze = 0
pending_silver = 0
pending_gold = 0

current_layer = None
for table_name, id_column, check_column, layer in tables_to_verify:
    # Print layer header
    if layer != current_layer:
        print(f"\n[{layer.upper()} LAYER]")
        print("-" * 70)
        current_layer = layer
    
    short_name = table_name.replace('acubed.ffr.', '').replace('`', '')
    
    if spark.catalog.tableExists(table_name):
        # Filter by appropriate column
        if id_column == "song_name":
            test_data = spark.table(table_name).filter(col(id_column) == test_song_name)
        else:
            test_data = spark.table(table_name).filter(col(id_column) == test_song_id)
        
        if test_data.count() > 0:
            if check_column:
                current_value = test_data.select(check_column).first()[check_column]
                
                # Check if still has fake value
                is_fake = (current_value == fake_version) or (current_value == 999)
                
                # For timestamp columns, check if they have the fake offset (original + 999999)
                if check_column == "timestamp":
                    # Can't easily check original values, but we can verify data exists
                    is_fake = False  # Assume processed if data exists
                    print(f"✓ {short_name:40} - data exists ({test_data.count()} records)")
                    success_count += 1
                elif is_fake:
                    if layer == "bronze":
                        print(f"❌ {short_name:40} - NOT updated (still {current_value})")
                        failure_count += 1
                    elif layer == "silver":
                        print(f"⏳ {short_name:40} - awaiting silver update (still {current_value})")
                        pending_silver += 1
                    elif layer == "gold":
                        print(f"⏳ {short_name:40} - awaiting gold update (still {current_value})")
                        pending_gold += 1
                else:
                    print(f"✅ {short_name:40} - updated (now {current_value})")
                    success_count += 1
            else:
                # Just verify song exists
                print(f"✓ {short_name:40} - song exists ({test_data.count()} records)")
                success_count += 1
        else:
            if layer == "bronze":
                print(f"❌ {short_name:40} - song NOT found")
                failure_count += 1
            elif layer == "silver":
                print(f"⏳ {short_name:40} - not yet populated")
                pending_silver += 1
            elif layer == "gold":
                print(f"⏳ {short_name:40} - not yet populated")
                pending_gold += 1
    else:
        print(f"⊘ {short_name:40} - table doesn't exist")
        if layer == "bronze":
            failure_count += 1
        elif layer == "silver":
            pending_silver += 1
        elif layer == "gold":
            pending_gold += 1

print("\n" + "="*70)
print("VERIFICATION SUMMARY")
print("="*70)
print(f"\n✅ Success: {success_count} checks passed")
print(f"❌ Failed: {failure_count} checks failed")
if pending_silver > 0:
    print(f"⏳ Silver pending: {pending_silver} tables awaiting silver notebook execution")
if pending_gold > 0:
    print(f"⏳ Gold pending: {pending_gold} tables awaiting gold notebook execution")

# Determine overall test status
if failure_count == 0 and success_count > 0:
    if pending_silver > 0 or pending_gold > 0:
        print(f"\n✅ PARTIAL SUCCESS")
        
        if pending_bronze == 0 and pending_silver > 0:
            print(f"\nBronze layer updates completed successfully:")
            print(f"  ✓ Detected song {test_song_id} with stale data")
            print(f"  ✓ Identified it as updated (swf_version mismatch)")
            print(f"  ✓ Updated it via Delta MERGE operation")
            print(f"  ✓ Restored correct API values")
            
            print(f"\n📋 Next Steps:")
            print(f"  • Run silver layer notebooks to propagate updates")
            if pending_gold > 0:
                print(f"  • Run gold layer notebooks to propagate to final layer")
            print(f"  • Re-run this verification to confirm full propagation")
        elif pending_gold > 0:
            print(f"\nBronze and Silver layer updates completed successfully!")
            print(f"\n📋 Next Steps:")
            print(f"  • Run gold layer notebooks to propagate updates")
            print(f"  • Re-run this verification to confirm full propagation")
    else:
        print(f"\n🎉 FULL PIPELINE TEST PASSED")
        print(f"\nUpdates successfully propagated through all layers:")
        print(f"  ✓ Bronze layer (API sync)")
        print(f"  ✓ Silver layer (feature recomputation)")
        print(f"  ✓ Gold layer (final aggregations)")
        print(f"\nSong {test_song_id} now has correct values in all {success_count} tested tables")
        print(f"\n✨ End-to-end update detection and propagation working correctly!")
elif failure_count > 0:
    print(f"\n❌ TEST FAILED")
    print(f"\nDiagnostics:")
    print(f"  • Did you run the bronze ingestion notebook after Step 2?")
    print(f"  • Check bronze ingestion Cell 3 output - did it show 'Updated songs: 1+' ?")
    print(f"  • Look for any error messages in the MERGE operation")
    print(f"  • Verify the update detection logic in bronze ingestion Cell 3")
    print(f"  • Ensure song {test_song_id} exists in source tables before modification")
else:
    print(f"\n⚠️ Could not verify - no data to check")
    print(f"   Ensure tables exist and song {test_song_id} is present")
    print(f"   Run Step 1 and Step 2 first to set up test data")